In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Dense, Embedding, Input, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# --- 1. Prepare the Dataset ---
corpus = [
    "the quick brown fox jumps over the lazy dog",
    "the quick brown dog jumps over the lazy fox",
    "a man is walking on the street",
    "a dog is barking at the cat",
    "the street is busy with people",
]

# --- 2. Tokenize and Create Word-to-Index Mappings ---
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
print(f"Vocabulary Size: {vocab_size}")

# --- 3. Generate CBOW Training Data ---
sequences = tokenizer.texts_to_sequences(corpus)
window_size = 2
data = []

for sequence in sequences:
    for i, word in enumerate(sequence):
        # Create a list for context words (left and right within window)
        context = []
        for j in range(i - window_size, i + window_size + 1):
            if j != i and 0 <= j < len(sequence):
                context.append(sequence[j])
        if context:
            data.append((context, word))

# Separate context arrays (X) and target word arrays (y)
X_context = [item[0] for item in data]
y_target = np.array([item[1] for item in data])

# Pad the context sequences to ensure a uniform shape matrix
max_len = max(len(x) for x in X_context)
X_context_padded = pad_sequences(X_context, maxlen=max_len, padding="post")
y_target_one_hot = to_categorical(y_target, num_classes=vocab_size)

# --- 4. Build the CBOW Model (Functional API) ---
embedding_dim = 100

# Input placeholder layer
input_context = Input(shape=(max_len,), name="Context_Input")

# Embedding layer to convert word indices to dense continuous vectors
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    name="Word_Embeddings",
)(input_context)

# Lambda layer to average the context embedding vectors
lambda_layer = Lambda(
    lambda x: K.mean(x, axis=1),
    output_shape=(embedding_dim,),
    name="Average_Context",
)(embedding_layer)

# Output classification layer layer using Softmax activation
output_layer = Dense(
    vocab_size, activation="softmax", name="Target_Prediction"
)(lambda_layer)

# Create and define structural architecture bounds
cbow_model = Model(inputs=input_context, outputs=output_layer)

# --- 5. Compile and Train the Model ---
cbow_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
cbow_model.summary()

print("\nTraining the CBOW model...")
cbow_model.fit(X_context_padded, y_target_one_hot, epochs=500, verbose=0)
print("Training complete.")

# --- 6. Extract and Visualise Word Embeddings ---
# Safely fetch the embedding matrix weights by name layer reference
embedding_matrix = cbow_model.get_layer("Word_Embeddings").get_weights()[0]

print("\nLearned Word Embeddings (First 5 Dimensions shown):")
for word, idx in list(word_index.items())[:10]:  # Show first 10 words
    if idx < vocab_size:
        print(f"Word: '{word:<10}' Vector: {embedding_matrix[idx][:5]}")


Vocabulary Size: 21


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Context_Input (InputLayer)      │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Word_Embeddings (Embedding)     │ (None, 4, 100)         │         2,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Average_Context (Lambda)        │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Target_Prediction (Dense)       │ (None, 21)             │         2,121 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,221 (16.49 KB)

 Trainable params: 4,221 (16.49 KB)

 Non-trainable params: 0 (0.00 B)


Training the CBOW model...
Training complete.

Learned Word Embeddings (First 5 Dimensions shown):
Word: 'the       ' Vector: [-0.35136116 -0.6038424  -0.21534088 -0.4171397  -0.15397371]
Word: 'dog       ' Vector: [-0.21470346 -0.09641796  0.32758492  0.16445407 -0.1093529 ]
Word: 'is        ' Vector: [ 0.52428573  0.63180876 -0.35929146  0.3919534  -0.49042946]
Word: 'quick     ' Vector: [ 0.15081938  0.03434935 -0.2633968  -0.1119181   0.3497599 ]
Word: 'brown     ' Vector: [-0.45495567 -0.2836564   0.44070402 -0.11117887 -0.5177411 ]
Word: 'fox       ' Vector: [-0.3424249  -0.09043816  0.3968605  -0.02200267  0.16411164]
Word: 'jumps     ' Vector: [ 0.11234034 -0.09267887 -0.3880011   0.25221667  0.56159467]
Word: 'over      ' Vector: [-0.43212578 -0.13755898  0.17512567 -0.4429726  -0.03203014]
Word: 'lazy      ' Vector: [-0.05812351 -0.16752571 -0.05469906  0.43319923  0.4219541 ]
Word: 'a         ' Vector: [ 0.49616474  0.42751676 -0.71985286  0.11151565 -0.6908606 ]
